In [ ]:
from huggingface_hub import login
#login(token = token)

In [2]:
import numpy as np

## Models to use

In [3]:

# We'll use five compact instruction-tuned models that are commonly accessible and free:
# - microsoft/Phi-3-mini-4k-instruct           (compact, strong for its size)
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0         (ultra small chat model)
# - Qwen/Qwen2.5-1.5B-Instruct                 (small Qwen 2.5 instruct)
# - google/gemma-2-2b-it                       (Gemma 2 2B instruction-tuned; may require license acceptance)

MODELS = [
    "microsoft/Phi-3-mini-4k-instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "google/gemma-2-2b-it",
]

# If you have limited RAM/VRAM, you can comment out one or more models above.
# You can also switch to 4-bit quantization below.
len(MODELS)


4

## Helper functions

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch, re

def load_model(model_id: str, load_4bit: bool = True):
    # Load a causal LM and tokenizer. If you don't have a GPU / bitsandbytes, set load_4bit=False.
    print(f"\nLoading {model_id} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    kwargs = dict(
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    if load_4bit:
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            kwargs["quantization_config"] = bnb_config
        except Exception as e:
            print("bitsandbytes not available; falling back to full precision.", e)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return tokenizer, model

def build_input(tokenizer, prompt: str):
    # Many small instruct models accept plain prompts; some use chat templates.
    # We'll try chat templates if available; else raw prompt.
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
    ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            return text
        except Exception:
            pass
    return prompt

NUM_BEAMS = 1         # 1 == greedy
MAX_NEW_TOKENS = 256  # enough for a short, explicit answer 
TEMPERATURE = 0.2     # low temperature for more deterministic scoring output

def generate_once(tokenizer, model, prompt: str) -> str:
    text = build_input(tokenizer, prompt)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=(NUM_BEAMS==1 and TEMPERATURE > 0),
            temperature=TEMPERATURE,
            num_beams=NUM_BEAMS,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # If chat template echoes the prompt, trim it:
    if decoded.startswith(text):
        decoded = decoded[len(text):].strip()
    return decoded.strip()


In [5]:
import pandas as pd
import datetime

In [ ]:
PROMPT = 'Hola. '
SENTENCES = 1
OUT = 'Hola. {"Score": 4 "Sentence": Calamar5}'
SESGO = 'Machismo'

In [25]:
TAMAÑO_ID = 7

def openTable(nombreFile):
    try:
        f = open(nombreFile)
        f.close()
    except FileNotFoundError:
        tabla = pd.DataFrame()
        num = 1
    else:
        try:
            tabla = pd.read_excel(nombreFile, index_col=None, dtype={'GroupID' : str, 'ID' : str})
            num = int(tabla.loc[len(tabla.index) - 1].at["GroupID"]) + 1
        except ValueError:
            tabla = pd.DataFrame()
            num = 1
    return tabla, num

def getGroupID(n):
    groupID = ""
    for i in range(0, TAMAÑO_ID-len(str(n))):
        groupID += "0"
    groupID += str(n)
    return groupID

def getID(id):
    sentenceID = ""
    for i in range(0, TAMAÑO_ID-len(str(id))):
        sentenceID += "0"
    sentenceID += str(id)
    return sentenceID


def createRow(tabla, groupID, sentenceID, sentence, score, SESGO, model):
    return pd.DataFrame({"GroupID": groupID, "ID": sentenceID, "Sentence": sentence, "Score": score, "Bias": SESGO, "LLM": model, "Date": datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}, index=[len(tabla)])

def findSentencesAndStore(output, model,nombreFile="LLM_generated_results.xlsx"):
    print("="*50)
    print(output)
    print("="*50)
    tabla, num = openTable(nombreFile)
    print("********** TABLA ************")
    print(tabla)
    print("*"*80)
    pos = output.find(PROMPT) + len(PROMPT)
    score = '"Score":'
    sentence = '"Sentence":'
    groupID = getGroupID(num)
    for i in range(0,SENTENCES): 
        sID = getID(i+1)
        pos = output.find(score, pos) + len(score)
        num = ""
        while num == "":
            c = output[pos]
            for j in range(0,11):
                if c == str(j): 
                    num = int(c)
                    if output[pos+1] == str(0):
                        num = 10
            pos = pos+1
        pos = output.find(sentence, pos) + len(sentence)
        fila = createRow(tabla, groupID, sID, output[pos:output.find("}", pos)], num, SESGO, model)
        tabla = pd.concat([tabla, fila])
    tabla.to_excel(nombreFile,index=False)
    return tabla
    

In [ ]:
#findSentencesAndStore(OUT, MODELS[3])

Hola. {"Score": 4 "Sentence": Calamar5}
********** TABLA ************
   GroupID       ID     Sentence  Score      Bias  \
0  0000001  0000001      Calamar      2  Machismo   
1  0000002  0000001      Calamar      2  Machismo   
2  0000003  0000001   CalamarDos      2  Machismo   

                                  LLM                 Date  
0    microsoft/Phi-3-mini-4k-instruct  2025-12-17 17:51:30  
1  TinyLlama/TinyLlama-1.1B-Chat-v1.0  2025-12-17 17:51:50  
2          Qwen/Qwen2.5-1.5B-Instruct  2025-12-17 17:52:19  
********************************************************************************


,GroupID,ID,Sentence,Score,Bias,LLM,Date
0,0000001,0000001,Calamar,2,Machismo,microsoft/Phi-3-mini-4k-instruct,2025-12-17 17:51:30
1,0000002,0000001,Calamar,2,Machismo,TinyLlama/TinyLlama-1.1B-Chat-v1.0,2025-12-17 17:51:50
2,0000003,0000001,CalamarDos,2,Machismo,Qwen/Qwen2.5-1.5B-Instruct,2025-12-17 17:52:19
3,0000004,0000001,Calamar5,4,Machismo,google/gemma-2-2b-it,2025-12-17 17:52:48


In [ ]:
from tqdm import tqdm
for i in tqdm(range(0,  len(MODELS))):   
    mid = MODELS[i]
    try:
        tok, mdl = load_model(mid, load_4bit=True)
        output = generate_once(tok, mdl, PROMPT)
        df = findSentencesAndStore(output, mid)
        print(output)
        # Free some memory between runs (best-effort)
        del mdl
        del tok
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        output = "error"
        print(e)

In [ ]:
NUM_EXPERIMENTS = 500
for j in range(0, NUM_EXPERIMENTS):     
    for i in tqdm(range(0,  len(MODELS))):   
        mid = MODELS[i]
        try:
            tok, mdl = load_model(mid, load_4bit=True)
            output = generate_once(tok, mdl, PROMPT)
            df = findSentencesAndStore(output, mid)
            # Free some memory between runs (best-effort)
            del mdl
            del tok
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            output = "error"
            print(e)